<a href="https://colab.research.google.com/github/sreenadhyadav883/7006SCN2526MAYJUL-7006SCN_SYA_16073319/blob/main/notebooks/Task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Run this cell once at the start of a fresh Colab runtime
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
!pip install -q -U "pyspark[connect]~=4.0.0" findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
import findspark

# Java path used by Colab after installing OpenJDK 17
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
findspark.init()
from pyspark.sql import SparkSession

# Stop an old Spark session if the cell is re-run
try:
    spark.stop()
except NameError:
    pass
spark = (
    SparkSession.builder
    .appName("7006SCN_PySpark")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark

In [3]:
# Loading the dataset from the public mirror
# Used -q (quiet) so it doesn't spam the googlecolab output with download logs
!wget -q https://datasets-documentation.s3.eu-west-3.amazonaws.com/amazon_reviews/amazon_reviews_2015.snappy.parquet

In [4]:
import time
from pyspark.sql.functions import col, when
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
print("Task 4 Performance Experiment...")

#Reloading baseline cleaned 1% sample data
df = spark.read.parquet("amazon_reviews_2015.snappy.parquet")
clean_df = df.dropna(subset=["review_body", "star_rating"])
clean_df = clean_df.withColumn("review_body", col("review_body").cast("string"))
final_df = clean_df.withColumn("label", when(col("star_rating") >= 4, 1).otherwise(0))
sampled_df = final_df.sample(withReplacement=False, fraction=0.01, seed=42)

#NLP transformation pipeline steps
tokenizer = Tokenizer(inputCol="review_body", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashing_tf = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=5000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
pipeline = Pipeline(stages=[tokenizer, remover, hashing_tf, idf])
ml_data = pipeline.fit(sampled_df).transform(sampled_df)
train_data, test_data = ml_data.randomSplit([0.8, 0.2], seed=42)

#Baseline model initialization
lr = LogisticRegression(featuresCol="features", labelCol="label", regParam=0.1)
print("partition scalability testing setup done!")

Task 4 Performance Experiment...
partition scalability testing setup done!


In [5]:
# Test 1:Low Parallelism(2 Partitions)
print("Running Experiment A: Repartitioning data to 2 partitions...")
train_data_2 = train_data.repartition(2)
train_data_2.cache()
train_data_2.count()
start_time = time.time()
model_2 = lr.fit(train_data_2)
time_2_partitions = time.time() - start_time
print(f"Total training Time with 2 Partitions is : {time_2_partitions:.2f} seconds")
train_data_2.unpersist() # Clear memory


#Test 2:Higher Parallelism(8 Partitions)
print("\nRunning Test 2: Repartitioning data to 8 partitions...")
train_data_8 = train_data.repartition(8)
train_data_8.cache()
train_data_8.count() # Force materialization of cache
start_time = time.time()
model_8 = lr.fit(train_data_8)
time_8_partitions = time.time() - start_time
print(f"Training Time with 8 Partitions: {time_8_partitions:.2f} seconds")
train_data_8.unpersist() # Clear memory

print("\nThe performance test is completed !...")
print(f"Time Difference: {abs(time_2_partitions - time_8_partitions):.2f} seconds")

Running Experiment A: Repartitioning data to 2 partitions...
Total training Time with 2 Partitions is : 14.86 seconds

Running Test 2: Repartitioning data to 8 partitions...
Training Time with 8 Partitions: 12.23 seconds

The performance test is completed !...
Time Difference: 2.62 seconds
